# 01 — Injection functions: build and test

Tests the three fault-injection functions (ground truth for the modules). Functions live in `src/injection.py`.

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import numpy as np, pandas as pd
from injection import inject_anomalies, inject_schema_drift, inject_missing_values

In [2]:
DATA_PATH = '../data/HI-Small_Trans.csv'   # <-- set to your HPC path
df = pd.read_csv(DATA_PATH, nrows=100_000)
print('shape:', df.shape); print('columns:', list(df.columns)); df.head(3)

shape: (100000, 11)
columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0


## Test 1 — inject_anomalies

In [3]:
corr, gt = inject_anomalies(df, 'Amount Paid', rate=0.02, multiplier=50, seed=42)
idx = gt[gt].index[:5]
print('flagged:', gt.sum(), '| expected:', int(len(df)*0.02))
print('x50 ok:', np.allclose(corr.loc[idx,'Amount Paid'], df.loc[idx,'Amount Paid']*50))

flagged: 2000 | expected: 2000
x50 ok: True


## Test 2 — inject_missing_values

In [4]:
corr2, gt2 = inject_missing_values(df, 'Receiving Currency', rate=0.10, seed=42)
n=corr2['Receiving Currency'].isna().sum()
print('NaNs:', n, '| expected:', int(len(df)*0.10), '| match:', n==int(len(df)*0.10))

NaNs: 10000 | expected: 10000 | match: True


## Test 3 — inject_schema_drift

In [5]:
d1,l1=inject_schema_drift(df, drop_column='Payment Format')
d2,l2=inject_schema_drift(df, rename_map={'Amount Paid':'amt_paid'})
d3,l3=inject_schema_drift(df, dtype_change=('From Bank', str))
print('drop:', 'Payment Format' not in d1.columns, l1)
print('rename:', 'amt_paid' in d2.columns, l2)
print('dtype:', d3['From Bank'].dtype, l3)

drop: True {'dropped_column': 'Payment Format'}
rename: True {'renamed_columns': {'Amount Paid': 'amt_paid'}}
dtype: str {'dtype_changed': {'From Bank': <class 'str'>}}


## Test 4 — reproducibility

In [6]:
_,a=inject_anomalies(df,'Amount Paid',seed=42)
_,b=inject_anomalies(df,'Amount Paid',seed=42)
print('same seed identical:', a.equals(b))

same seed identical: True
